In [1]:
import os

from azure.ai.formrecognizer import FormRecognizerClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

ID_FORM_RECOGNIZER_ENDPOINT = os.environ["ID_RECOGNIZER_ENDPOINT"]
ID_FORM_RECOGNIZER_KEY = os.environ["ID_RECOGNIZER_KEY"]

In [3]:
def extract_id_info(local_image_path: str):
    form_recognizer_client = FormRecognizerClient(endpoint=ID_FORM_RECOGNIZER_ENDPOINT,
                                                  credential=AzureKeyCredential(ID_FORM_RECOGNIZER_KEY))
    with open(local_image_path, "rb") as f:
        poller = form_recognizer_client.begin_recognize_identity_documents(identity_document=f)
        return poller.result()


def extract_id_card_fields(identity_card):
    field_mappings = [
        ("FirstName", "First Name"),
        ("LastName", "Last Name"),
        ("DocumentNumber", "Document Number"),
        ("DateOfBirth", "Date of Birth"),
        ("DateOfExpiration", "Date of Expiration"),
        ("Sex", "Sex"),
        ("Address", "Address"),
        ("CountryRegion", "Country/Region"),
        ("Region", "Region"),
    ]

    extracted_fields = []
    for field_key, field_label in field_mappings:
        field = identity_card.fields.get(field_key)
        if field:
            extracted_fields.append((field_label, field.value, field.confidence))

    return extracted_fields


def print_id_card_fields(extracted_fields):
    for field_label, field_value, field_confidence in extracted_fields:
        print("{}: {}".format(field_label, field_value))


def get_id_card_details(identity_card):
    extracted_fields = extract_id_card_fields(identity_card)
    print_id_card_fields(extracted_fields)

In [4]:
local_image_path = r"..\material_preparation_step\digital-ids\ca-dl-sameer-kumar.png"
collected_id_cards = extract_id_info(local_image_path)

for index_id, id_card in enumerate(collected_id_cards, start=1):
    print(f"Displaying identity card details ....... # {index_id}")
    get_id_card_details(id_card)
    print("---------------- EOL -------------------------\n")

Displaying identity card details ....... # 1
First Name: Sameer
Last Name: Kumar
Document Number: D4556673
Date of Birth: 1990-01-25
Date of Expiration: 2025-08-28
Sex: M
Country/Region: USA
Region: California
---------------- EOL -------------------------

